# Final Project: Building a Rainfall Prediction Classifier

This notebook completes the 14 coding exercises for a binary classifier that predicts heavy rainfall (`HeavyRain`: 1 = precipitation > 20 mm, 0 = otherwise).

**Before running:** place `rainfall.csv` in the same folder as this notebook, or update `DATA_PATH` in Exercise 2 to its location. Run the cells from top to bottom.


## Exercise 1: Import Libraries
Import the libraries needed for data handling, visualisation, preprocessing, modelling, and evaluation.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay
)

%matplotlib inline


## Exercise 2: Load the Dataset
Load `rainfall.csv` and check that the expected target column is present.

In [ ]:
DATA_PATH = "rainfall.csv"  # Change this if the CSV is stored elsewhere.
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

if "HeavyRain" not in df.columns:
    raise ValueError("The dataset must contain a 'HeavyRain' target column (1 = heavy rain, 0 = no heavy rain).")


## Exercise 3: Exploratory Data Analysis (EDA)
Inspect the data types, summary statistics, missing values, and target distribution.

In [ ]:
df.info()
display(df.describe(include="all").T)
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing values"))

sns.countplot(data=df, x="HeavyRain")
plt.title("Heavy Rain Occurrence")
plt.xlabel("HeavyRain (0 = no heavy rain, 1 = heavy rain)")
plt.show()


## Exercise 4: Handle Missing Values
Fill numeric columns with their median and categorical columns with their most frequent value.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

for col in numeric_cols:
    if col != "HeavyRain":
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    mode_values = df[col].mode(dropna=True)
    if not mode_values.empty:
        df[col] = df[col].fillna(mode_values.iloc[0])

print("Missing values remaining:", int(df.isna().sum().sum()))


## Exercise 5: Feature Engineering
Create 3-day rolling averages for temperature and humidity when matching columns are available. If the dataset uses different column names, the code will report that rather than failing.

In [ ]:
def find_column(candidates, columns):
    lower_map = {str(c).lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    return None

temp_col = find_column(["Temperature", "Temp", "MeanTemp", "Temp3pm", "MaxTemp"], df.columns)
humidity_col = find_column(["Humidity", "Humidity3pm", "Humidity9am"], df.columns)

if temp_col:
    df["Temp_3day_avg"] = df[temp_col].rolling(window=3, min_periods=1).mean()
else:
    print("Temperature feature not found; Temp_3day_avg was not created.")

if humidity_col:
    df["Hum_3day_avg"] = df[humidity_col].rolling(window=3, min_periods=1).mean()
else:
    print("Humidity feature not found; Hum_3day_avg was not created.")

display(df.head())


## Exercise 6: Define Features and Target
Separate predictors (`X`) from the binary target (`y`). Remove rows with missing target values and ensure the target is binary.

In [ ]:
target = "HeavyRain"
df = df.dropna(subset=[target]).copy()
X = df.drop(columns=[target])
y = df[target].astype(int)

if y.nunique() != 2:
    raise ValueError("HeavyRain must contain exactly two classes, coded 0 and 1.")

print("Feature matrix:", X.shape)
print("Target distribution:")
display(y.value_counts().sort_index().rename(index={0:"No heavy rain (0)", 1:"Heavy rain (1)"}).to_frame("Count"))


## Exercise 7: Train-Test Split
Create an 80/20 stratified split so the class distribution is preserved in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## Exercise 8: Build a Random Forest Pipeline
Scale numeric features and one-hot encode categorical features, then apply a Random Forest classifier.

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="drop"
)

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("rf", RandomForestClassifier(random_state=42, class_weight="balanced"))
])
rf_pipeline


## Exercise 9: Baseline Random Forest Training
Fit the baseline pipeline on the training data.

In [ ]:
rf_pipeline.fit(X_train, y_train)
print("Baseline Random Forest model trained.")


## Exercise 10: Evaluate the Baseline Model
Calculate accuracy, classification report, confusion matrix, true positive rate, and ROC-AUC; then plot the ROC curve.

In [ ]:
y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_cm = confusion_matrix(y_test, y_pred_rf, labels=[0, 1])
rf_tpr = (rf_cm[1, 1] / (rf_cm[1, 1] + rf_cm[1, 0]) * 100
          if (rf_cm[1, 1] + rf_cm[1, 0]) else 0.0)
rf_roc_auc = roc_auc_score(y_test, y_proba_rf)

print(f"Random Forest Accuracy: {rf_acc:.4f} ({rf_acc:.1%})")
print(f"Random Forest True Positive Rate: {rf_tpr:.0f}%")
print("Confusion Matrix (rows = actual [0,1], columns = predicted [0,1]):\n", rf_cm)
print(classification_report(y_test, y_pred_rf, zero_division=0))
print(f"Random Forest ROC-AUC: {rf_roc_auc:.4f}")

RocCurveDisplay.from_predictions(y_test, y_proba_rf)
plt.title("Random Forest ROC Curve")
plt.show()


## Exercise 11: Hyperparameter Tuning
Use 5-fold GridSearchCV with ROC-AUC scoring to search over the number of trees and maximum depth.

In [ ]:
param_grid = {
    "rf__n_estimators": [100, 200],
    "rf__max_depth": [None, 10, 20]
}
grid = GridSearchCV(
    rf_pipeline, param_grid, cv=5, scoring="roc_auc", n_jobs=-1, refit=True
)
grid.fit(X_train, y_train)
print("Best Random Forest parameters:", grid.best_params_)
print(f"Best 5-fold CV ROC-AUC: {grid.best_score_:.4f}")


## Exercise 12: Retrain and Evaluate the Tuned Model
Use the best estimator found by GridSearchCV and evaluate it on the held-out test set.

In [ ]:
best_rf = grid.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)
y_proba_best_rf = best_rf.predict_proba(X_test)[:, 1]

best_rf_acc = accuracy_score(y_test, y_pred_best_rf)
best_rf_cm = confusion_matrix(y_test, y_pred_best_rf, labels=[0, 1])
best_rf_tpr = (best_rf_cm[1, 1] / (best_rf_cm[1, 1] + best_rf_cm[1, 0]) * 100
               if (best_rf_cm[1, 1] + best_rf_cm[1, 0]) else 0.0)
best_rf_roc_auc = roc_auc_score(y_test, y_proba_best_rf)

print(f"Tuned Random Forest Accuracy: {best_rf_acc:.4f} ({best_rf_acc:.1%})")
print(f"Tuned Random Forest True Positive Rate: {best_rf_tpr:.0f}%")
print("Tuned Random Forest Confusion Matrix:\n", best_rf_cm)
print(classification_report(y_test, y_pred_best_rf, zero_division=0))
print(f"Tuned Random Forest ROC-AUC: {best_rf_roc_auc:.4f}")


## Exercise 13: Cross-Validation Score
Report the mean 10-fold ROC-AUC for the tuned model on the full dataset.

In [ ]:
cv_scores = cross_val_score(best_rf, X, y, cv=10, scoring="roc_auc", n_jobs=-1)
print("10-fold ROC-AUC scores:", np.round(cv_scores, 4))
print(f"Mean 10-fold ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


## Exercise 14: Save the Trained Model
Save the final trained pipeline to `rainfall_classifier.pkl` in the current folder.

In [ ]:
MODEL_PATH = "rainfall_classifier.pkl"
joblib.dump(best_rf, MODEL_PATH)
print("Model saved to:", os.path.abspath(MODEL_PATH))


## Optional: Compare with Logistic Regression

This extra section supports the assignment question comparing accuracy and true positive rate for Logistic Regression and Random Forest. Use the measured results below rather than assuming which model is better.


In [ ]:
logreg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("logreg", __import__("sklearn.linear_model", fromlist=["LogisticRegression"]).LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ))
])
logreg_pipeline.fit(X_train, y_train)

y_pred_lr = logreg_pipeline.predict(X_test)
y_proba_lr = logreg_pipeline.predict_proba(X_test)[:, 1]
lr_acc = accuracy_score(y_test, y_pred_lr)
lr_cm = confusion_matrix(y_test, y_pred_lr, labels=[0, 1])
lr_tpr = (lr_cm[1, 1] / (lr_cm[1, 1] + lr_cm[1, 0]) * 100
          if (lr_cm[1, 1] + lr_cm[1, 0]) else 0.0)
lr_roc_auc = roc_auc_score(y_test, y_proba_lr)

print(f"Logistic Regression Accuracy: {lr_acc:.4f} ({lr_acc:.1%})")
print(f"Logistic Regression True Positive Rate: {lr_tpr:.0f}%")
print("Logistic Regression Confusion Matrix:\n", lr_cm)
print(f"Logistic Regression ROC-AUC: {lr_roc_auc:.4f}")


## Metrics Summary and Conclusion
This table uses the actual results from the notebook run. The model with the higher ROC-AUC has better overall ability to separate the two classes; accuracy and true positive rate should also be considered.

In [ ]:
summary = pd.DataFrame({
    "Model": ["Tuned Random Forest", "Logistic Regression"],
    "Accuracy": [best_rf_acc, lr_acc],
    "True Positive Rate (%)": [best_rf_tpr, lr_tpr],
    "ROC-AUC": [best_rf_roc_auc, lr_roc_auc],
})
display(summary)

if best_rf_roc_auc > lr_roc_auc:
    print("Random Forest has the higher ROC-AUC on this test set.")
elif lr_roc_auc > best_rf_roc_auc:
    print("Logistic Regression has the higher ROC-AUC on this test set.")
else:
    print("Both models have the same ROC-AUC on this test set.")


### Important notes
- Run all cells from top to bottom after placing `rainfall.csv` in the same folder as the notebook.
- This notebook expects a binary target column named `HeavyRain` coded as 0 and 1.
- Results depend on the actual dataset; metrics are calculated when the notebook is run and are not hard-coded.
